In [ ]:
# ── Standard library ─────────────────────────────────────────────────────────
import os                               
import json, csv                       
import re                              

# ── Data handling ─────────────────────────────────────────────────────────────
import numpy as np                      
import pandas as pd                     

# ── Lyric scraping ────────────────────────────────────────────────────────────
import lyricsgenius                     # Genius API wrapper for pulling song lyrics

# ── Model and training ────────────────────────────────────────────────────────
from transformers import (
    LongformerTokenizer,                # tokenizer for Longformer's sliding-window attention
    LongformerForSequenceClassification,# Longformer with a classification head for theme labeling
    TrainingArguments,                  # hyperparameter and training configuration
    Trainer                             # handles training loop, eval, and checkpointing
)

import torch                            # tensor operations and GPU management
from transformers import pipeline       # zero-shot classification for auto-labeling
from sklearn.model_selection import train_test_split  # splits dataset into train/val sets
from sklearn.metrics import f1_score, roc_auc_score   # multi-label evaluation metrics

# ── Dataset utilities ─────────────────────────────────────────────────────────
from torch.utils.data import Dataset   # base class for the custom LyricsDataset
import ast                              # safely parses stringified lists from CSV columns

# ── Hugging Face authentication ───────────────────────────────────────────────
# Authenticates with Hugging Face Hub to enable pushing model weights after training.
from huggingface_hub import login
from dotenv import load_dotenv
import os

In [29]:
load_dotenv()

# Verify the Hugging Face and Genius tokens loaded correctly before trying to login
hf_token = os.environ.get("HF_TOKEN")
if hf_token is None:
    raise ValueError("HF_TOKEN not found — check your .env file is in the project root")


genius_token = os.environ.get("GENIUS_TOKEN")
if genius_token is None:
    raise ValueError("GENIUS_TOKEN not found — check your .env file is in the project root")

login(token=hf_token)
print("Hugging Face login successful")
print("Genius login successful")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful
Genius login successful


These settings help prevent GPU context corruption and improve error reporting during training

In [30]:
# CUDA and GPU Environment Configuration

import os  # Included to keep cell self-contained

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:512" # Limit GPU memory growth so crashes don't corrupt the context

# Functions

In [ ]:
# ── Functions for Song Searching and Processing ────────────────────────────────────────────────
def clean_lyrics(raw_lyrics):
    """
    Strips metadata and standardizes raw Genius lyrics into clean plain text.
    This function removes that metadata along with section headers, extra
    whitespace, punctuation, and capitalisation.

    Args:
        raw_lyrics (str): Raw lyrics string returned by genius.search_song()

    Returns:
        str: Cleaned, lowercased lyrics ready for tokenization
    """
    
    # Genius prepends metadata before the first section header e.g. [Intro]
    starter = ']' # Finding the first ']' and slicing from there drops all prepended metadata
    position = raw_lyrics.find(starter)
    lyrics = raw_lyrics[position+1:]

    # Remove section headers like [Chorus], [Verse 1], etc.
    lyrics = re.sub(r"\[.*?\]", "", lyrics)

    # Remove extra whitespace, newlines
    lyrics = re.sub(r"\n+", " ", lyrics)
    lyrics = re.sub(r"\s+", " ", lyrics)

    # Remove punctuation
    lyrics = re.sub(r"[^\w\s']", "", lyrics)

    # Lowercase
    lyrics = lyrics.lower()

    return lyrics


def song_search(song,artist):
    """
    Fetches and cleans lyrics for a given song from the Genius API.

    Initialises a Genius API client, searches for the song by name and
    artist, then passes the raw lyrics through clean_lyrics() before
    returning them.

    Args:
        song (str):   Title of the song to search for
        artist (str): Name of the artist who recorded the song

    Returns:
        str: Cleaned lyrics string
    """

    genius = lyricsgenius.Genius(genius_token, timeout=60)
    song = genius.search_song(song, artist)
    lyrics = song.lyrics

    # Handles if song is not found
    if song is None:
        print(f"Could not find lyrics for '{song}' by {artist}")
        return None
    
    cleaned_lyrics = clean_lyrics(lyrics)
    return cleaned_lyrics




# ── Per-song labeling function ───────────────────────────────────────────────────────────────
def label_lyrics(lyrics: str) -> tuple[list[str], dict]:
    """
    Assigns theme labels to a single song's lyrics using zero-shot classification.

    Short lyrics (under 400 words) are passed directly to the classifier.
    Long lyrics are split into 400-word chunks to respect bart-large-mnli's
    1024 token input limit, scored independently, then averaged to produce
    a single set of theme scores for the song.

    Args:
        lyrics (str): Cleaned lyrics string for a single song

    Returns:
        tuple: (
            assigned (list[str]): themes scoring above THRESHOLD,
            avg_scores (dict):    full score map for all themes — useful
                                  for threshold tuning and inspection
        )
    """

    words = lyrics.split()
    chunk_size = 400

    # Only chunk lyrics that exceed the safe token limit — avoids unnecessary splitting overhead for songs under 400 words
    if len(words) <= chunk_size:
        chunks = [lyrics]
    else:
        chunks = [
            " ".join(words[i:i + chunk_size])
            for i in range(0, len(words), chunk_size)
        ]

    # Accumulate scores for each theme across all chunks
    score_map = {theme: 0.0 for theme in THEMES}
    for chunk in chunks:
        result = zero_shot(chunk, THEMES, multi_label=True)
        for label, score in zip(result["labels"], result["scores"]):
            score_map[label] += score

    # Average scores across chunks so song length doesn't inflate scores.
    # For songs under 400 words n=1, so averaging has no effect.
    n = len(chunks)
    avg_scores = {t: score_map[t] / n for t in THEMES}

    # Assign all themes that meet the confidence threshold
    assigned = [t for t, s in avg_scores.items() if s >= THRESHOLD]

    # Fallback — always assign at least the top scoring theme to
    # prevent empty label lists which would break multi-label training
    if not assigned:
        assigned = [max(avg_scores, key=avg_scores.get)]

    return assigned, avg_scores


# ── Evaluation metrics ──────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    """
    Computes multi-label classification metrics after each validation epoch.

    Args:
        eval_pred: (logits, labels) tuple provided by Trainer

    Returns:
        dict with three metrics:
            f1_micro  — overall performance weighted by label frequency
            f1_macro  — average performance treating all labels equally
            roc_auc   — ranking quality across all labels and thresholds
    """

    logits, labels = eval_pred
    probs = torch.sigmoid(torch.tensor(logits)).numpy()
    preds = (probs >= 0.5).astype(int)
    return {
        "f1_micro":  f1_score(labels, preds, average="micro",  zero_division=0),
        "f1_macro":  f1_score(labels, preds, average="macro",  zero_division=0),
        "roc_auc":   roc_auc_score(labels, probs, average="macro"),
    }


def predict_themes(lyrics: str, threshold: float = 0.5) -> dict:
    """
    Predicts themes present in a song's lyrics using the fine-tuned Longformer.

    Tokenizes the input lyrics, runs a forward pass through the model, and
    converts raw logits to per-theme confidence scores using sigmoid activation.
    Only themes scoring above the threshold are returned.

    Args:
        lyrics (str):       cleaned lyrics string — run through clean_lyrics() before passing in to match training data format
        threshold (float):  minimum confidence score for a theme to be included in results. Defaults to 0.5 — lower this to surface more themes, raise it (e.g. 0.7) for higher precision predictions only

    Returns:
        dict: themes scoring above threshold sorted by confidence score
              returns empty dict if no themes exceed the threshold
    """
    model.eval()
    inputs = tokenizer(
        lyrics,
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).to(model.device)

    global_attn = torch.zeros_like(inputs["input_ids"])
    global_attn[:, 0] = 1
    inputs["global_attention_mask"] = global_attn

    with torch.no_grad():
        logits = model(**inputs).logits

    probs = torch.sigmoid(logits).squeeze().cpu().numpy()
    results = {THEMES[i]: round(float(probs[i]), 3) for i in range(NUM_LABELS)}

    predicted = {t: s for t, s in results.items() if s >= threshold}
    return dict(sorted(predicted.items(), key=lambda x: -x[1]))

In [32]:
# Lyrics are stored in a .txt file with comma delimiters for convenience
lyrics_df = pd.read_csv(
    'lyrics_list.txt', 
    delimiter=',',
    header=None
)
lyrics_df.columns = ['lyrics','Nan'] # Assigns column names. Second column is always empty due to trailing comma
lyrics_df = lyrics_df.drop(labels='Nan',axis=1) # Drops empty column

In [33]:
lyrics_df

,lyrics
0,hey jude don't make it bad take a sad song and...
1,when i find myself in times of trouble mother...
2,yesterday all my troubles seemed so far away ...
3,shoot me shoot me shoot me shoot me here come...
4,it's been a hard day's night and i've been wo...
...,...
904,you took me back but you shouldn't have now i...
905,i've been waiting i've been waiting for this ...
906,time is never worth my time blue shine bleeds...
907,i could run i could cry i could plead i could...


In [34]:
lyrics_df["word_count"] = lyrics_df["lyrics"].apply(lambda x: len(x.split()))
print(lyrics_df["word_count"].describe())

count     909.000000
mean      312.970297
std       160.370978
min        27.000000
25%       207.000000
50%       271.000000
75%       363.000000
max      1351.000000
Name: word_count, dtype: float64


# Assigning classifications to lyrics list

This section automatically assigns theme labels to each song in the dataset using facebook/bart-large-mnli, a zero-shot NLI classifier that scores each candidate theme against the lyrics without requiring any training data.

Long lyrics (over 400 words) are split into chunks and scores are averaged across chunks to stay within the model's 1024 token input limit. Shortlyrics are passed directly without splitting overhead.

Outputs a labeled JSON file used as training data.

In [ ]:
# ── Model setup ───────────────────────────────────────────────────────────────
zero_shot = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=0  # Runs on GPU
)

# ── Themes ─────────────────────────────────────────────────────────────────────
# These themes cover the most common lyrical subject matters but can be adjusted to affect what the model learns to detect
THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]

THRESHOLD = 0.20  # Minimum confidence score for a theme to be assigned to a song

# ── Run labeling over full dataset ────────────────────────────────────────────
assigned_labels = []
all_scores = []

for i, row in lyrics_df.iterrows():
    labels, scores = label_lyrics(row["lyrics"])
    assigned_labels.append(labels)
    all_scores.append(scores)

    # Progress update every 50 songs — useful for monitoring long runs
    if i % 50 == 0:
        print(f"Processed {i}/{len(lyrics_df)} — last labels: {labels}")

# Adds results to DataFrame
lyrics_df["themes"] = assigned_labels
lyrics_df["theme_scores"] = all_scores


# Save as JSON — avoids the list serialization issues that CSV causes
# with multi-label columns (the ast.literal_eval problem from earlier)
lyrics_df.to_json("lyrics_labeled.json", orient="records", lines=True)
print(f"Done. {len(lyrics_df)} songs labeled and saved to lyrics_labeled.json")

Device set to use cuda:0


Processed 0/909 — last labels: ['love and romance', 'heartbreak and loss', 'identity and self-discovery', 'rebellion and resistance', 'nostalgia and memory', 'depression and mental health', 'social commentary', 'celebration and joy', 'spirituality and faith', 'ambition and success', 'loneliness and isolation', 'death and mortality']


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Processed 50/909 — last labels: ['heartbreak and loss', 'identity and self-discovery', 'rebellion and resistance', 'nostalgia and memory', 'depression and mental health', 'social commentary', 'ambition and success', 'loneliness and isolation', 'death and mortality']
Processed 100/909 — last labels: ['love and romance', 'heartbreak and loss', 'identity and self-discovery', 'rebellion and resistance', 'nostalgia and memory', 'depression and mental health', 'social commentary', 'celebration and joy', 'spirituality and faith', 'ambition and success', 'loneliness and isolation', 'death and mortality']
Processed 150/909 — last labels: ['love and romance', 'identity and self-discovery', 'rebellion and resistance', 'nostalgia and memory', 'depression and mental health', 'social commentary', 'celebration and joy', 'spirituality and faith', 'ambition and success', 'loneliness and isolation', 'death and mortality']
Processed 200/909 — last labels: ['love and romance', 'heartbreak and loss', 'iden

# Fine-Tuning

Fine-tunes allenai/longformer-base-4096 on the auto-labeled lyrics dataset.The model learns to predict multiple themes per song simultaneously using a multi-hot label vector and binary cross-entropy loss.

In [36]:
# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME  = "allenai/longformer-base-4096"
MAX_LENGTH  = 2048   # safe limit for 16GB VRAM — covers 95%+ of lyrics
BATCH_SIZE  = 2      # kept low to prevent CUDA out-of-memory crashes
GRAD_ACCUM  = 8      # effective batch size = 16
EPOCHS      = 6      # enough epochs for convergence on ~900 songs
LR          = 2e-5   # standard fine-tuning learning rate for BERT-family models

THEMES = [
    "love and romance", "heartbreak and loss", "identity and self-discovery",
    "rebellion and resistance", "nostalgia and memory", "depression and mental health",
    "social commentary", "celebration and joy", "spirituality and faith",
    "ambition and success", "loneliness and isolation", "death and mortality",
]
NUM_LABELS = len(THEMES)
THEME_TO_IDX = {t: i for i, t in enumerate(THEMES)} # Maps each theme string to its index in the label vector

# ── Dataset class ────────────────────────────────────────────────────────────
class LyricsDataset(Dataset):
    """
    PyTorch Dataset for tokenized lyrics and their multi-hot theme labels.

    Tokenizes all lyrics at initialization and stores them in memory.
    Each item returns a dict of tensors ready for the Longformer model.

    Args:
        lyrics_list (list[str]):  cleaned lyrics strings
        labels_list (list[list]): list of theme label lists per song
        tokenizer:                LongformerTokenizer instance
    """

    def __init__(self, lyrics_list, labels_list, tokenizer):
        self.encodings = []
        self.labels = []

        for lyrics, themes in zip(lyrics_list, labels_list):
            enc = tokenizer(
                lyrics,
                max_length=MAX_LENGTH,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
            )
            
            # Longformer uses two attention types:
            # - Local (sliding window): every token attends to its neighbours
            # - Global: designated tokens attend to all other tokens
            # [CLS] token (index 0) must have global attention for classification tasks — it aggregates the full sequence into a single representation
            global_attn = torch.zeros(MAX_LENGTH, dtype=torch.long)
            global_attn[0] = 1 # Enable global attention on [CLS] only

            self.encodings.append({
                "input_ids":            enc["input_ids"].squeeze(),
                "attention_mask":       enc["attention_mask"].squeeze(),
                "global_attention_mask": global_attn,
            })

            # Build multi-hot label vector of length NUM_LABELS.
            # Each assigned theme sets its corresponding index to 1.0.
            label_vec = torch.zeros(NUM_LABELS)
            for theme in themes:
                if theme in THEME_TO_IDX:
                    label_vec[THEME_TO_IDX[theme]] = 1.0
            self.labels.append(label_vec)

    def __len__(self):
        return len(self.labels) # Required by PyTorch DataLoader to determine dataset size

    def __getitem__(self, idx):
        return {**self.encodings[idx], "labels": self.labels[idx]} # Returns a single tokenized song and its label vector

NOTE: Model weights have been saved to Hugging Face Hub and do not need to be retrained. This cell is kept for documentation and reproducibility. To load the trained model for inference see the inference cell below.

In [37]:
# =============================================================================
# ARCHIVED — Training code below does not need to be re-run.
# Model weights are saved at: your-username/lyric-theme-longformer on Hugging Face Hub. To load for inference see the inference cell.
# =============================================================================

# ── Load & split data ────────────────────────────────────────────────────────
lyrics_df = pd.read_json("lyrics_labeled.json", lines=True)

# Split into 85% training / 15% validation
train_df, val_df = train_test_split(lyrics_df, test_size=0.15, random_state=42)

tokenizer = LongformerTokenizer.from_pretrained(MODEL_NAME)

train_dataset = LyricsDataset(
    train_df["lyrics"].tolist(), 
    train_df["themes"].tolist(), 
    tokenizer
)
val_dataset = LyricsDataset(
    val_df["lyrics"].tolist(),
    val_df["themes"].tolist(),
    tokenizer
)

# ── Model ─────────────────────────────────────────────────────────────────────
# Loads pretrained Longformer weights and attaches a randomly initialised classification head for NUM_LABELS outputs — expected warning on first load
model = LongformerForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification", # Uses sigmoid + BCE loss
)

# Reduces VRAM usage by recomputing activations during backward pass instead of storing them — trades speed for memory stability
model.gradient_checkpointing_enable()


# ── Training Config ──────────────────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./lyric-theme-longformer",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,     
    learning_rate=LR,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    fp16=False,                                   # disabled — caused cuBLAS errors on this hardware
    bf16=True,
    gradient_checkpointing=True,
    dataloader_num_workers=0,
    no_cuda=False,
    logging_steps=25,
    report_to="none",
)

# ── Training ──────────────────────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save final model and tokenizer locally as a backup
trainer.save_model("./lyric-theme-longformer/best")
tokenizer.save_pretrained("./lyric-theme-longformer/best")
print("Training complete.")

Some weights of LongformerForSequenceClassification were not initialized from the model checkpoint at allenai/longformer-base-4096 and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,Roc Auc
1,0.648900,0.423271,0.877741,0.778006,nan
2,0.408300,0.392299,0.897455,0.852234,nan
3,0.364200,0.344048,0.906227,0.879593,nan
4,0.323500,0.331110,0.912040,0.895436,nan
5,0.297200,0.324711,0.912996,0.894097,nan
6,0.285200,0.323613,0.911540,0.892754,nan


c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metrics\_ranking.py:424: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\metric

Training complete.


This cell is for pulling model from local path

In [38]:
# Model Loading for Inference

# Load from the best checkpoint that was saved during training
LOCAL_PATH = "./lyric-theme-longformer/best"

model = LongformerForSequenceClassification.from_pretrained(LOCAL_PATH)
tokenizer = LongformerTokenizer.from_pretrained(LOCAL_PATH)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set model to evaluation mode
model.eval()

print(f"Model loaded successfully — running on {device}")

Model loaded successfully — running on cuda


In [40]:
# Model Upload to Hugging Face
HF_REPO = "gpecorino/lyric-theme-longformer"

model.push_to_hub(HF_REPO)
tokenizer.push_to_hub(HF_REPO)
print(f"Model saved to https://huggingface.co/{HF_REPO}")

c:\Users\Monado\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Monado\.cache\huggingface\hub\models--gpecorino--lyric-theme-longformer. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Processing Files (1 / 1): 100%|██████████|  595MB /  595MB, 10.7MB/s  
New Data

Model saved to https://huggingface.co/gpecorino/lyric-theme-longformer


This cell is for pulling model from Hugging Face, if it is not already in the local path

In [41]:
# Model loading to Hugging Face
HF_REPO = "gpecorino/lyric-theme-longformer"

model = LongformerForSequenceClassification.from_pretrained(HF_REPO)
tokenizer = LongformerTokenizer.from_pretrained(HF_REPO)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set model to evaluation mode
model.eval()

print(f"Model loaded successfully — running on {device}")

Model loaded successfully — running on cuda


# Using the Model on New Songs

In [42]:
# Load in new song and run it through the model to predict themes
lyrics = song_search('Risk it All','Bruno Mars')
if lyrics:
    themes = predict_themes(lyrics)
    print(themes)

Searching for "Risk it All" by Bruno Mars...
Done.
{'social commentary': 0.951, 'identity and self-discovery': 0.944, 'nostalgia and memory': 0.933, 'love and romance': 0.918, 'celebration and joy': 0.91, 'rebellion and resistance': 0.809, 'depression and mental health': 0.713, 'spirituality and faith': 0.712, 'heartbreak and loss': 0.693, 'ambition and success': 0.691, 'death and mortality': 0.571, 'loneliness and isolation': 0.508}
